## Restauration CPU indépendante
Version Kaggle [v1](https://www.kaggle.com/code/marwanedrissi/industrialdefect-ai-patchcore-bottle-restore-check?scriptVersionId=368927596). Attacher MVTec AD et les Outputs du smoke test v3 dans une nouvelle session CPU. Les rapports et les sorties historiques sont ceux de Kaggle.

In [1]:
from pathlib import Path
import hashlib
import json

EXPECTED_SHA = (
    "37e3d731aca54d2f0272b79082d21ee"
    "afebc96faeda91d19a72a84353451a9fa"
)

checkpoints = list(
    Path("/kaggle/input").rglob("patchcore_model.pt")
)

print("Checkpoints found:", len(checkpoints))

for path in checkpoints:
    print(path)

assert len(checkpoints) == 1, (
    "Attach exactly one saved PatchCore notebook output"
)

checkpoint_path = checkpoints[0]
artifact_dir = checkpoint_path.parent

report_path = artifact_dir / "smoke_report.json"
reference_path = artifact_dir / "reference_prediction.npz"

assert report_path.is_file()
assert reference_path.is_file()

actual_sha = hashlib.sha256(
    checkpoint_path.read_bytes()
).hexdigest()

assert actual_sha == EXPECTED_SHA

report = json.loads(report_path.read_text())

print("\n=== SAVED ARTIFACTS ===")
print("Run ID:", report["run_id"])
print("Checkpoint:", checkpoint_path)
print("SHA256:", actual_sha)
print("Validation image:", report["validation_image"])

print("\nIndependent artifact access: OK")


Checkpoints found: 1
/kaggle/input/notebooks/marwanedrissi/industrialdefect-ai-patchcore-bottle-smoke/patchcore/bottle/smoke-001/patchcore_model.pt

=== SAVED ARTIFACTS ===
Run ID: patchcore-bottle-smoke-001
Checkpoint: /kaggle/input/notebooks/marwanedrissi/industrialdefect-ai-patchcore-bottle-smoke/patchcore/bottle/smoke-001/patchcore_model.pt
SHA256: 37e3d731aca54d2f0272b79082d21eeafebc96faeda91d19a72a84353451a9fa
Validation image: bottle/train/good/002.png

Independent artifact access: OK


In [2]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-q", "anomalib==2.7.0"
    ],
    check=True
)

print("Anomalib installed")


Anomalib installed


In [4]:

import importlib.metadata as metadata
from packaging.version import Version
import torch
import torchvision

print("=== RESTORATION ENVIRONMENT ===")

for package, expected in report["versions"].items():
    actual = metadata.version(package)

    if package in ("torch", "torchvision"):
        # Allow CPU and CUDA builds of the same version
        compatible = (
            Version(actual).base_version
            == Version(expected).base_version
        )
    else:
        compatible = (actual == expected)

    print(
        f"{package}: {actual} "
        f"(expected: {expected})"
        f" -> {'OK' if compatible else 'MISMATCH'}"
    )

    assert compatible, f"Incompatible {package}"

print("\n=== DEVICE ===")
print("CUDA available:", torch.cuda.is_available())
print("CUDA runtime:", torch.version.cuda)

DEVICE = torch.device("cpu")
print("Restoration device:", DEVICE)

print("\nEnvironment validation: OK")


=== RESTORATION ENVIRONMENT ===
torch: 2.11.0+cpu (expected: 2.11.0+cu130) -> OK
torchvision: 0.26.0+cpu (expected: 0.26.0+cu130) -> OK
anomalib: 2.7.0 (expected: 2.7.0) -> OK
timm: 1.0.30 (expected: 1.0.30) -> OK
lightning: 2.6.6 (expected: 2.6.6) -> OK

=== DEVICE ===
CUDA available: False
CUDA runtime: None
Restoration device: cpu

Environment validation: OK


In [7]:
import subprocess
import sys
import hashlib
from pathlib import Path

REPO_DIR = Path("/kaggle/working/industrial-defect-ai")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/MRdrissi/industrial-defect-ai.git",
            str(REPO_DIR)
        ],
        check=True
    )

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", report["git_commit"]
    ],
    check=True
)

sys.path.insert(0, str(REPO_DIR))

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform

dataset_root = Path(
    "/kaggle/input/datasets/ipythonx/mvtec-ad"
)

manifest = (
    REPO_DIR / "docs/data/us-02-03/bottle_manifest.csv"
)

assert dataset_root.is_dir()
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == (
    report["manifest_sha256"]
)

val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest,
    category="bottle",
    split="val",
    transform=MVTecTransform(
        size=(256, 256),
        profile="patchcore"
    )
)

target_path = report["validation_image"]

matches = [
    i for i in range(len(val_dataset))
    if val_dataset[i]["image_path"] == target_path
]

assert len(matches) == 1

sample = val_dataset[matches[0]]

print("=== VALIDATION IMAGE ===")
print("Path:", sample["image_path"])
print("Shape:", tuple(sample["image"].shape))
print("Label:", sample["label"])
print("Validation image restored: OK")

Cloning into '/kaggle/working/industrial-defect-ai'...
HEAD is now at 0a6f5fa merge: integrate EPIC-01 and EPIC-02 into develop


=== VALIDATION IMAGE ===
Path: bottle/train/good/002.png
Shape: (3, 256, 256)
Label: 0
Validation image restored: OK


### Recharger et prédire
La cellule suivante rétablit le chargement et l’inférence absents de l’export Kaggle. Elle ne porte aucune sortie historique. La cellule suivante compare sa prédiction à la référence GPU sauvegardée.

In [ ]:
import numpy as np
from anomalib.models.image.patchcore.torch_model import PatchcoreModel

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
restored_model = PatchcoreModel(
    layers=checkpoint["layers"],
    backbone=checkpoint["backbone"],
    pre_trained=False,
    num_neighbors=checkpoint["num_neighbors"],
).to(DEVICE)
restored_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
restored_model.feature_extractor.requires_grad_(False)
restored_model.eval()

with np.load(reference_path, allow_pickle=False) as reference:
    expected_score = torch.from_numpy(reference["pred_score"]).to(DEVICE)
    expected_map = torch.from_numpy(reference["anomaly_map"]).to(DEVICE)

with torch.inference_mode():
    prediction = restored_model(sample["image"].unsqueeze(0).to(DEVICE))
actual_score = prediction.pred_score
actual_map = prediction.anomaly_map


In [9]:

import torch
import json
from pathlib import Path

print("=== INDEPENDENT RESTORATION VALIDATION ===")

# 1. Shape and numerical validity
assert actual_score.shape == expected_score.shape
assert actual_map.shape == expected_map.shape
assert torch.isfinite(actual_score).all()
assert torch.isfinite(actual_map).all()

# 2. Absolute numerical differences
score_abs_diff = (
    actual_score - expected_score
).abs().max().item()

map_abs_error = (
    actual_map - expected_map
).abs()

map_max_diff = map_abs_error.max().item()
map_mean_diff = map_abs_error.mean().item()

# 3. Difference relative to reference map range
map_range = (
    expected_map.max() - expected_map.min()
).item()

assert map_range > 0

map_relative_range_error = map_max_diff / map_range

print("GPU score:", expected_score.item())
print("CPU score:", actual_score.item())
print("Score absolute difference:", score_abs_diff)

print("\nMap max absolute difference:", map_max_diff)
print("Map mean absolute difference:", map_mean_diff)
print(
    "Maximum error / map range (%):",
    100 * map_relative_range_error
)

# 4. Explicit acceptance thresholds
SCORE_ABS_TOL = 1e-3
MAP_ABS_TOL = 1e-3
MAP_RANGE_REL_TOL = 1e-4

assert score_abs_diff <= SCORE_ABS_TOL
assert map_max_diff <= MAP_ABS_TOL
assert map_relative_range_error <= MAP_RANGE_REL_TOL

# 5. Confirm memory bank restoration
assert tuple(restored_model.memory_bank.shape) == (
    819, 1536
)

print("\n=== FINAL RESULT ===")
print("Checkpoint loaded: OK")
print("Memory Bank restored: OK")
print("CPU inference: OK")
print("Numerical consistency: OK")
print("\nINDEPENDENT RESTORATION: PASS")


=== INDEPENDENT RESTORATION VALIDATION ===
GPU score: 30.64944839477539
CPU score: 30.64940071105957
Score absolute difference: 4.76837158203125e-05

Map max absolute difference: 0.00045299530029296875
Map mean absolute difference: 3.481773819657974e-05
Maximum error / map range (%): 0.0016449888306989961

=== FINAL RESULT ===
Checkpoint loaded: OK
Memory Bank restored: OK
CPU inference: OK
Numerical consistency: OK

INDEPENDENT RESTORATION: PASS


In [10]:

import json
import hashlib
import importlib.metadata as metadata
from pathlib import Path

RESTORE_DIR = Path(
    "/kaggle/working/patchcore/bottle/restore-001"
)
RESTORE_DIR.mkdir(parents=True, exist_ok=True)

restore_report = {
    "run_id": "patchcore-bottle-restore-001",
    "source_run_id": report["run_id"],
    "restoration_device": "cpu",
    "checkpoint_sha256": hashlib.sha256(
        checkpoint_path.read_bytes()
    ).hexdigest(),
    "memory_bank_shape": list(
        restored_model.memory_bank.shape
    ),
    "validation_image": sample["image_path"],
    "torch_version": metadata.version("torch"),
    "anomalib_version": metadata.version("anomalib"),
    "reference_score": float(expected_score.item()),
    "restored_score": float(actual_score.item()),
    "score_absolute_difference": score_abs_diff,
    "map_max_absolute_difference": map_max_diff,
    "map_mean_absolute_difference": map_mean_diff,
    "map_max_error_relative_to_range": (
        map_relative_range_error
    ),
    "tolerances": {
        "score_absolute": SCORE_ABS_TOL,
        "map_absolute": MAP_ABS_TOL,
        "map_relative_to_range": MAP_RANGE_REL_TOL
    },
    "memory_bank_recomputed": False,
    "strict_state_dict_loading": True,
    "restoration_status": "PASS"
}

assert (
    restore_report["checkpoint_sha256"]
    == report["checkpoint_sha256"]
)

output_path = RESTORE_DIR / "restore_report.json"

output_path.write_text(
    json.dumps(restore_report, indent=2),
    encoding="utf-8"
)

print("Restore report:", output_path)
print("Checkpoint integrity: OK")
print("Restore evidence saved: OK")


Restore report: /kaggle/working/patchcore/bottle/restore-001/restore_report.json
Checkpoint integrity: OK
Restore evidence saved: OK
